In [ ]:
import torch

torch.cuda.empty_cache()

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def _no_norm_forward_transform(
    self, inputs: torch.Tensor, patched_pads: torch.Tensor
) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
    """Input is of shape [B, N, P]."""
    mu = torch.tensor([0.0], device=inputs.device)
    sigma = torch.tensor([1.0], device=inputs.device)
    outputs = inputs  # no normalization
    return outputs, (mu, sigma)


def _no_norm_reverse_transform(
    self, outputs: torch.Tensor, stats: tuple[torch.Tensor, torch.Tensor]
) -> torch.Tensor:
    return outputs  # no normalization, so reverse is identity


def _global_forward_transform(
    self, inputs: torch.Tensor, patched_pads: torch.Tensor
) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
    """Input is of shape [B, N, P]."""
    masked_inputs = inputs.masked_fill(patched_pads.bool(), float("nan"))
    mu = torch.nanmean(masked_inputs).unsqueeze(0)
    sigma = torch.sqrt(torch.nanmean((masked_inputs - mu) ** 2)).unsqueeze(0)
    sigma = torch.where(
        sigma < self.config.tolerance,
        torch.tensor(1.0, dtype=sigma.dtype, device=sigma.device),
        sigma,
    )

    # Normalize each patch
    outputs = (inputs - mu[:, None, None]) / sigma[:, None, None]

    # clamp norm output to max value to prevent instability
    outputs = torch.where(
        torch.abs(inputs - self.config.pad_val) < self.config.tolerance,
        torch.tensor(self.config.pad_val, dtype=outputs.dtype, device=outputs.device),
        outputs,
    )
    return outputs, (mu, sigma)


def _global_reverse_transform(
    self, outputs: torch.Tensor, stats: tuple[torch.Tensor, torch.Tensor]
) -> torch.Tensor:
    mu, sigma = stats
    return outputs * sigma[:, None, None, None] + mu[:, None, None, None]


def _instancenorm_forward_transform(
    self, inputs: torch.Tensor, patched_pads: torch.Tensor
) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
    """Input is of shape [B, N, P]."""
    masked_inputs = inputs.masked_fill(patched_pads.bool(), float("nan"))
    mu = torch.nanmean(masked_inputs).unsqueeze(0)
    sigma = torch.sqrt(torch.nanmean((masked_inputs - mu) ** 2)).unsqueeze(0)
    sigma = torch.where(
        sigma < self.config.tolerance,
        torch.tensor(1.0, dtype=sigma.dtype, device=sigma.device),
        sigma,
    )
    # print("mu", mu.item(), "sigma", sigma.item())
    # if inputs.max() > 1000.0:
    #     print("Warning: Input values are very large, which may cause instability in normalization.")
    #     print("Input stats - min:", inputs.min().item(), "max:", inputs.max().item(), "mean:", inputs.mean().item(), "std:", inputs.std().item())

    # Normalize
    scaled_inputs = (inputs - mu[:, None, None]) / sigma[:, None, None]

    # Apply arcsinh transformation
    # print("scaled inputs", scaled_inputs.min(), scaled_inputs.max())

    outputs = torch.arcsinh(
        scaled_inputs.masked_fill(patched_pads.bool(), 0.0)
    )  # arcsinh(0) = 0, so this won't affect the padded values

    # print("arcsinh outputs", outputs.min(), outputs.max())

    outputs = torch.where(
        torch.abs(inputs - self.config.pad_val) < self.config.tolerance,
        torch.tensor(self.config.pad_val, dtype=outputs.dtype, device=outputs.device),
        outputs,
    )
    # print("final inputs", outputs.min(), outputs.max())

    # print(outputs.shape, mu.shape, scale.shape)

    return outputs, (mu, sigma)


def _instancenorm_reverse_transform(
    self, outputs: torch.Tensor, stats: tuple[torch.Tensor, torch.Tensor]
) -> torch.Tensor:
    mu, sigma = stats
    # print("mu", mu.item(), "sigma", sigma.item())
    # Inverse arcsinh transformation
    # print("outputs before inverse transform", outputs.min(), outputs.max())
    scaled_inputs = torch.sinh(outputs)
    # print("scaled outputs after inverse transform", scaled_inputs.min(), scaled_inputs.max())
    # Denormalize
    inputs_f32 = scaled_inputs * sigma[:, None, None] + mu[:, None, None]
    return inputs_f32.to(outputs.dtype)

In [ ]:
LR = 1e-3
BATCH_SIZE = 128
RESULTS_DIR_BASE_NAME = "lora-norm-ablation"
MAX_STEPS = 500
EVAL_STEPS = 100
PREDICTION_LENGTH = 128
FULL_SUBSAMPLING = False
STRIDE = 58

In [ ]:
from fusiontimeseries.lib.config import FTSConfig

fts_config = FTSConfig(
    context_length=288,
    prediction_length=PREDICTION_LENGTH,
    sampling_stride=STRIDE,
    pred_tail_timestamps=80,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    lr_scheduler_type="constant",
    optimizer_type="adamw_torch_fused",
    max_grad_norm=1.0,
    max_steps=MAX_STEPS,
    eval_steps=EVAL_STEPS,
    gradient_accumulation_steps=1,
    full_subsampling=FULL_SUBSAMPLING,
    padding_value=0.0,  # chronos2 has NaN as padding value
    padding_mask_default=0.0,
    padding_mask_indicator=1.0,
    # sampling_strategy="tail_mean",
    # sampling_bins=5,
)

In [ ]:
from fusiontimeseries.ablations.dataset import BaselineTimeseriesDataset
# from fusiontimeseries.ablations.dataset import FTSAblationIterableDataset

TimeSeriesDataset = BaselineTimeseriesDataset

train_dataset, val_dataset = TimeSeriesDataset.train_val_split(fts_config)

In [ ]:
from timesfm import TimesFmCheckpoint, TimesFmHparams
from timesfm.timesfm_torch import TimesFmTorch
from timesfm.pytorch_patched_decoder import PatchedTimeSeriesDecoder


def get_model():

    repo_id = "google/timesfm-2.0-500m-pytorch"

    hparams = TimesFmHparams(
        backend=fts_config.device,  # type: ignore
        per_core_batch_size=fts_config.batch_size,
        horizon_len=fts_config.prediction_length,
        context_len=fts_config.context_length,
        num_layers=50,
        use_positional_embedding=False,
    )

    tfm = TimesFmTorch(
        hparams=hparams, checkpoint=TimesFmCheckpoint(huggingface_repo_id=repo_id)
    )

    model: PatchedTimeSeriesDecoder | None = tfm._model

    if model is None:
        raise ValueError("Model is None")

    return model


model: PatchedTimeSeriesDecoder = get_model()

In [ ]:
# patch the model's normalization functions to no-op for the base ablation
PatchedTimeSeriesDecoder._forward_transform = _instancenorm_forward_transform
PatchedTimeSeriesDecoder._reverse_transform = _instancenorm_reverse_transform

In [ ]:
# patch config to be HF Trainer compatible

from timesfm.pytorch_patched_decoder import TimesFMConfig
from fusiontimeseries.ablations.trainer import patch_times_fm_config

patch_times_fm_config(TimesFMConfig)

In [ ]:
from fusiontimeseries.loralib.layers import Linear

model = Linear.convert(
    module=model,
    kind="LoRA",
    lora_rank=8,
    lora_alpha=16,
    target_module_names=None,  # slap LoRA on all linear layers
)

In [ ]:
from pathlib import Path

from fusiontimeseries.lib.get_next_path import get_next_path

base_dir = Path("./newresults")
base_dir.mkdir(parents=True, exist_ok=True)
output_dir = get_next_path(base_fname=RESULTS_DIR_BASE_NAME, base_dir=base_dir)
output_dir.mkdir(parents=True, exist_ok=False)
print(f"Output directory created at: {output_dir}")

In [ ]:
from fusiontimeseries.loralib.utils import mark_only_lora_as_trainable
from fusiontimeseries.loralib.utils import print_trainable_parameters

mark_only_lora_as_trainable(model=model, bias="none")
print_trainable_parameters(model, save_path=output_dir / "trainable_params.json")

In [ ]:
from transformers.training_args import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=fts_config.batch_size,
    per_device_eval_batch_size=fts_config.batch_size,
    learning_rate=fts_config.learning_rate,
    lr_scheduler_type=fts_config.lr_scheduler_type,
    optim=fts_config.optimizer_type,
    logging_strategy="steps",
    logging_steps=fts_config.eval_steps,
    disable_tqdm=False,
    report_to="none",
    max_steps=fts_config.max_steps,
    gradient_accumulation_steps=fts_config.gradient_accumulation_steps,
    dataloader_num_workers=0,
    tf32=False,
    bf16=False,
    save_only_model=True,
    prediction_loss_only=True,
    save_total_limit=2,
    save_strategy="steps",
    save_steps=fts_config.eval_steps,
    eval_strategy="steps",
    eval_steps=fts_config.eval_steps,
    load_best_model_at_end=True,  # keep last model since validation set is quite unexpressive
    metric_for_best_model="eval_loss",
    use_cpu=False,
    label_names=[
        "future_target"
    ],  # must be truthy for HF Trainer to use overridden compute_loss
    remove_unused_columns=False,  # needed to not accidentally remove columns that our custom compute_loss relies on
    max_grad_norm=fts_config.max_grad_norm,
)
training_arguments._n_gpu = 1

In [ ]:
import json

from fusiontimeseries.ablations.trainer import TimesFMTrainer

trainer = TimesFMTrainer(
    model=model,
    train_args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    fts_config=fts_config,
)
with open(output_dir / "training_args.json", "w") as f:
    json.dump(trainer.args.to_dict(), f, indent=4)
fts_config.save_config(output_dir / "fts_config.json")

In [ ]:
import json
from fusiontimeseries.loralib.utils import lora_state_dict

train_output = trainer.train()

with open(output_dir / "train_summary.json", "w") as f:
    json.dump(train_output._asdict(), f, indent=4)

lora_weights = lora_state_dict(model)
torch.save(lora_weights, output_dir / "lora_weights.pt")

In [ ]:
benchmark_data = TimeSeriesDataset.get_benchmark_flux_traces(fts_config)
model = model.eval()

In [ ]:
id_results = TimeSeriesDataset.evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["id"],
    start_context_length=80,
)
round(id_results["rmse"].item(), 4), round(id_results["rmse_standard_error"].item(), 4)

In [ ]:
ood_results = TimeSeriesDataset.evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["ood"],
    start_context_length=80,
)
(
    round(ood_results["rmse"].item(), 4),
    round(ood_results["rmse_standard_error"].item(), 4),
)

In [ ]:
import matplotlib.pyplot as plt

print(id_results["forecasts"].keys())

for IDX in [115, 131, 148, 235, 262, 8]:
    plt.figure(figsize=(8, 6))
    plt.plot(id_results["ground_truths"][IDX], label="Ground Truth")
    plt.plot(id_results["forecasts"][IDX], label="Forecast")
    plt.axvline(x=80, color="gray", linestyle="--", label="Context-Prediction Boundary")
    plt.title("ID Forecast vs Ground Truth")
    plt.xlabel("Time Steps")
    plt.ylabel("Flux")
    plt.legend()
    plt.savefig(output_dir / f"{IDX}_id_forecast.png")

In [ ]:
import matplotlib.pyplot as plt

for IDX in ood_results["forecasts"].keys():
    plt.figure(figsize=(8, 6))
    plt.plot(ood_results["ground_truths"][IDX], label="Ground Truth")
    plt.plot(ood_results["forecasts"][IDX], label="Forecast")
    plt.axvline(x=80, color="gray", linestyle="--", label="Context-Prediction Boundary")
    plt.title("OOD Forecast vs Ground Truth")
    plt.xlabel("Time Steps")
    plt.ylabel("Flux")
    plt.legend()
    plt.savefig(output_dir / f"{IDX}_ood_forecast.png")

# Ablation Studies

## Timeseries Augmentation

- Basic: LR 5e-4, effective bs 128, pred len 128, max steps 4000, eval steps 400, noticed: val set gets worse from the start onwards.
- Basic LoRA:                                                                                               15.54 +- 5.13; 7.69 +- 1.68
- Change: fully-subsample 241 training samples -> Train/Val split: 723 (prev. 241) / 3 time series.         16.36 +- 5.71; 3.94 +- 1.21
- Change: lower stride to 1 to have more samples -> problem only two timeseries in one batch (2 * 59)       17.11 +- 6.08; 5.41 +- 1.71
- Change: random cropping ->                                                                                18.47 +- 6.06; 4.14 +- 1.04

During evaluation on test set we also start from 80, two-split strategy starts from 80 (and 138), tailoring the model to give better forecasts from 80 onwards.

## Sampling Strategies

- Tail-mean sampling 14.26 +- 4.92; 7.57 +- 1.39

In [ ]:
id_results = TimeSeriesDataset.evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["id"],
    start_context_length=202,
)

In [ ]:
import numpy as np


for forecast, target in zip(
    id_results["forecasts"].values(), id_results["ground_truths"].values()
):
    print(
        f"Forecast last 80 mean: {np.mean(forecast[-80:]):.4f}, Target last 80 mean: {np.mean(target[-80:]):.4f}"
    )

In [ ]:
len(id_results["forecasts"][115][:-80])

In [ ]:
from fusiontimeseries.lib.benchmarking import rmse_with_standard_error
from fusiontimeseries.lib.conditioning import ConditionRegistry
from fusiontimeseries.lib.dataset import FluxData


def evaluate_model_on_test_data(
    model,
    config: FTSConfig,
    benchmark_data: dict[int, FluxData],
    start_context_length: int,
) -> dict[str, float | dict[int, list[float]]]:
    forecasts: dict[int, list[float]] = {}
    ground_truths: dict[int, list[float]] = {}
    for flux_id, flux_data in benchmark_data.items():
        flux_data: FluxData
        energy_flux = np.array(flux_data.energy_flux)
        op_params = (
            torch.Tensor(flux_data.operating_parameters).unsqueeze(0).to(config.device)
        )

        ctx: np.ndarray = energy_flux[:start_context_length]
        while len(ctx) < len(energy_flux):
            with torch.no_grad():
                context_start_idx = config.context_length - len(ctx)
                tctx = torch.full(
                    size=(1, config.context_length),
                    fill_value=config.padding_value,
                )
                tctx[0, context_start_idx:] = torch.tensor(ctx)
                context_mask = torch.full_like(
                    tctx, fill_value=config.padding_mask_default
                )  # 0.0
                # context_mask[:cutoff_idx] = self.config.padding_mask_indicator context_masks.append(context_mask)
                context_mask[0, :context_start_idx] = (
                    config.padding_mask_indicator
                )  # 1.0

                with ConditionRegistry.patch(op_params=op_params):
                    with torch.autocast(device_type=config.device, dtype=torch.float32):
                        # predictions shape: (batch_size, num_patches, horizon_len, num_quantiles + 1(mean))
                        predictions: torch.Tensor = model(
                            tctx.to(config.device, non_blocking=True),
                            context_mask.to(config.device, non_blocking=True).float(),
                            torch.tensor([[0.0]], dtype=torch.long).to(
                                config.device, non_blocking=True
                            ),
                        )
            forecast = predictions[0, -1, : config.prediction_length, 0].cpu().numpy()
            ctx = np.concatenate([ctx, forecast])

        forecasts[flux_id] = ctx[: len(energy_flux)].tolist()
        ground_truths[flux_id] = flux_data.energy_flux

    benchmark_means: list[np.floating] = [
        np.mean(gt[-config.pred_tail_timestamps :]) for gt in ground_truths.values()
    ]
    forecast_means: list[np.floating] = [
        np.mean(fc[-config.pred_tail_timestamps :]) for fc in forecasts.values()
    ]
    rmse, rmse_se = rmse_with_standard_error(
        y_true=np.array(benchmark_means), y_pred=np.array(forecast_means)
    )
    return {
        "rmse": rmse,
        "rmse_standard_error": rmse_se,
        "forecasts": forecasts,
        "ground_truths": ground_truths,
    }

In [ ]:
id_results = evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["id"],
    start_context_length=80,
)
id_results["rmse"], id_results["rmse_standard_error"]

In [ ]:
ood_results = evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["ood"],
    start_context_length=80,
)
ood_results["rmse"], ood_results["rmse_standard_error"]

In [ ]:
# evaluate id and ood performance across different context lengths starting with 2, up to 202. Plot it

cl_rmse_results: list[tuple[int, float, float]] = []
for cl in range(22, 202 + 1, 20):
    id_results = evaluate_model_on_test_data(
        model=model,
        config=fts_config,
        benchmark_data=benchmark_data["id"],
        start_context_length=cl,
    )
    cl_rmse_results.append((cl, id_results["rmse"], id_results["rmse_standard_error"]))

In [ ]:
# save results to json
with open(output_dir / "cl_rmse_results.json", "w") as f:
    json.dump(
        [
            {"context_length": cl, "rmse": rmse, "rmse_standard_error": se}
            for cl, rmse, se in cl_rmse_results
        ],
        f,
        indent=4,
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

context_lengths = [x[0] for x in cl_rmse_results]
rmses = [x[1] for x in cl_rmse_results]
rmse_errors = [x[2] for x in cl_rmse_results]

plt.errorbar(
    context_lengths,
    rmses,
    yerr=rmse_errors,
    marker="o",
    capsize=2,
    alpha=1,
    color="pink",
    label="LoRA Fine-tuned",
)

plt.xticks(range(2, 203, 20))
plt.xlim(-12, 220)
plt.xlabel("Start Context Length")
plt.ylabel("RMSE ± Standard Error")
plt.title("ID Performance vs Context Length")
# plt.grid(axis="y")
# plt.legend()
plt.show()

# Hyperparameter seach (manual) DEPRECATED

500 steps, no lr scheduler, LoRA on every Linear Layer in the network, Trainable parameters: 5,223,424 / 504,052,384 (1.04%), GPU RAM ~8/16GB

|Batch Size (observed) | Learning Rate | ID RMSE +- SE | OOD RMSE +- SE| Prediction Length |
|-----------|---------------|---------------|---------------|-------------------|
|64(128)   | 1e-5          | 36.13 +- 9.86 | 21.21 +- 3.78 | 128 (base) |
|64(128)   | 1e-4          | 25.91 +- 7.40 | 13.33 +- 2.67 | 128 (base) |
|64(128)   | 5e-4          | 22.56 +- 8.24 | 10.09 +- 2.18 | 128 (base) |
|64(128)   | 1e-3          | 23.38 +- 7.20 | 8.01 +- 2.21 | 128 (base) |
|128(256)   | 1e-4          | 24.51 +- 6.01 | 15.62 +- 4.55 | 128 (base) |
|128(256)   | 5e-5          | 25.44 +- 6.48 | 16.48 +- 4.22 | 128 (base) |
|64(192)   | 5e-4          | 23.55 +- 7.46 | 10.74 +- 2.54 | 64 |

- 64 pred len: Sample future timestamps are ['80 - 144', '141 - 205', '202 - 266'] = 3 samples per sample
- 128 pred len (original timesfm): Sample future timestamps are ['80 - 208', '138 - 266'] = 2 samples per sample